# Feature Engineering — CRISP-DM Phase 3

This notebook documents feature engineering decisions and validates their impact on model performance.

**Features created:**
- `clv_estimate` — Customer Lifetime Value proxy
- `is_at_risk` — heuristic early-warning flag
- `engagement_score` — composite product usage metric
- `charge_to_tenure_ratio` — spending rate over lifetime
- `tenure_segment` — lifecycle stage label

In [ ]:
import sys; sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px

from src.data.generator import SyntheticDataGenerator
from src.features.engineering import FeatureEngineer
from src.features.selection import FeatureSelector
from src.data.preprocessor import DataPreprocessor, TARGET_COLUMN

In [ ]:
raw = SyntheticDataGenerator(42).generate(5_000)
engineer = FeatureEngineer()
df = engineer.fit_transform(raw)

new_cols = ['clv_estimate', 'monthly_charge_tier', 'is_at_risk',
            'engagement_score', 'tenure_segment', 'charge_to_tenure_ratio']
df[new_cols + ['churned']].head(10)

## Validate: do new features correlate with churn?

In [ ]:
numeric_new = ['clv_estimate', 'is_at_risk', 'engagement_score', 'charge_to_tenure_ratio']
corr = df[numeric_new + ['churned']].corr()['churned'].drop('churned').sort_values()
print(corr)
px.bar(corr, orientation='h', title='New Feature Correlation with Churn').show()

## `is_at_risk` flag accuracy

In [ ]:
at_risk_churn = df.groupby('is_at_risk')['churned'].mean()
print('Churn rate by is_at_risk:')
print(at_risk_churn)
# Expected: is_at_risk=1 should have significantly higher churn rate

## Feature Selection — remove redundant features

In [ ]:
y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN, 'customer_id',
                      'monthly_charge_tier', 'tenure_segment'], errors='ignore')

preprocessor = DataPreprocessor()
X_proc, _ = preprocessor.fit_transform(X)

selector = FeatureSelector(corr_threshold=0.90, importance_threshold=0.003)
X_selected = selector.fit_transform(X_proc, y)

print(f'Original features: {X_proc.shape[1]}')
print(f'After selection:   {X_selected.shape[1]}')
print(f'Dropped (corr):    {selector._dropped_corr_}')
print(f'Dropped (import.): {selector._dropped_importance_}')